<p style="text-align: center; margin-bottom: 0; font-size: 24px;">EAE1106 - Métodos Computacionais para Economia</p>
<p style="text-align: center; margin-bottom: 0;">Faculdade de Economia, Administração, Contabilidade e Atuária<br>Universidade de São Paulo</p>
<p style="text-align: center; margin-top: 5px;">1º semestre de 2026<br><br>Prof. Arthur Viaro</p>

---

# Aula 16: NumPy - Arrays e Operações Vetorizadas

- [Por que NumPy?](#motivacao)
- [O `ndarray`: a estrutura central](#ndarray)
- [Tipos de dados (`dtype`)](#dtype)
- [Criação de arrays](#criacao)
- [Arrays 2D (matrizes)](#arrays2d)
- [Indexação e fatiamento](#indexacao)
- [Views vs. cópias](#views)
- [Operações vetorizadas](#vetorizadas)
- [Funções estatísticas](#estatisticas)
- [Reshape e manipulação de forma](#reshape)

## Por que NumPy? <a id="motivacao"></a>

Economia lida constantemente com dados numéricos em grande escala: séries do IPCA mês a mês, indicadores de dezenas de países, preços de ações ao longo de anos. As listas Python são flexíveis, mas **inadequadas** para esse tipo de trabalho:

```python
inflacao = [0.42, 0.83, 0.16, 0.38, 0.44, 0.50,
            0.38, 0.44, 0.56, 0.61, 0.39, 0.52]

inflacao + 1      # ❌ TypeError — soma listas, não adiciona 1 a cada elemento
inflacao * 2      # ❌ duplica a lista — não multiplica os valores!
inflacao > 0.5    # ❌ TypeError — comparação elemento-a-elemento não existe
```

Para fazer o que queremos com listas, precisaríamos de laços `for` — lentos e verbosos. O **NumPy** (*Numerical Python*) resolve isso:

- **Vetorização**: operações matemáticas aplicadas a **arrays inteiros**, sem laços `for`
- **Velocidade**: código compilado em C → 10× a 100× mais rápido que Python puro
- **Ecossistema**: Pandas, SciPy, Matplotlib e scikit-learn são todos construídos sobre NumPy

> 💡 **Analogia**: lista Python é uma caixa onde qualquer objeto cabe. Array NumPy é uma régua de números do mesmo tipo — compacta, organizada e feita para cálculo.


In [ ]:
import numpy as np

## O `ndarray`: a estrutura central <a id="ndarray"></a>

O `ndarray` (*N-dimensional array*) é a estrutura de dados central do NumPy. Ele representa uma estrutura homogênea de dados - uma grade (ou sequeência) de **valores do mesmo tipo**, armazenados em posições consecutivas na memória.

Essa característica é o que torna o NumPy tão eficiente.

**Por que "mesmo tipo" faz diferença?** 

Em listas do Python, cada elemento é um objeto independente. Isso dá flexibilidade, mas tem um custo: o Python precisa gerenciar cada item separadamente, sem saber antecipadamente seu tamanho.

Já no `ndarray`, todos os elementos têm o mesmo tipo (por exemplo, `float64`) e ocupam o mesmo espaço em memória (8 bytes, nesse caso). Isso permite que o NumPy armazene os dados de forma compacta e execute operações em bloco (vetorização) utilizando rotinas de baixo nível altamente otimizadas (C/Fortran).

**Dimensões do array:**

O `ndarray` pode ter uma ou mais dimensões:

- **1D (vetor)**: usado para séries ou sequências - shape `(n,)`
- **2D (matriz)**: representa tabelas de dados - shape `(linhas, colunas)`
- **3D ou mais (tensores)**: estruturas mais complexas

Todo `ndarray` expõe atributos que descrevem sua estrutura:

| Atributo | O que informa | Array 1D `(12,)` | Array 2D `(5, 3)` |
|---|---|---|---|
| `.dtype` | Tipo dos elementos | `float64` | `float64` |
| `.shape` | Dimensões (formato do array) | `(12,)` | `(5, 3)` |
| `.ndim` | Número de dimensões | `1` | `2` |
| `.size` | Total de elementos | `12` | `15` |
| `.itemsize` | Tamanho (em bytes) de cada elemento | `8` | `8` |
| `.nbytes` | Memória total (`size × itemsize`) | `96 B` | `120 B` |


In [ ]:
# Array 1D: IPCA mensal de 2023 (% ao mês)
# Criado a partir de uma lista Python com np.array()
ipca = np.array([0.42, 0.83, 0.16, 0.38, 0.44, 0.50,
                 0.38, 0.44, 0.56, 0.61, 0.39, 0.52])

print("Array:", ...)
print()
print("dtype:    ", ...)    # todos os elementos são float64
print("shape:    ", ...)    # (12,) → 12 elementos, 1 dimensão
print("ndim:     ", ...)     # 1 dimensão
print("size:     ", ...)     # 12 elementos no total
print("nbytes:   ", ..., "bytes")  # 12 × 8 = 96 bytes

## Tipos de dados (`dtype`) <a id="dtype"></a>

O `dtype` (data type) define como cada elemento do array é armazenado em memória. Isso afeta diretamente três coisas

- memória (quantos bytes cada valor ocupa)
- precisão numérica
- performance

O NumPy **infere automaticamente** o tipo a partir dos valores, mas você pode (e muitas vezes deve) especificá-lo explicitamente.

Tipos mais comuns em economia:
- `float64`: ponto flutuante de 64 bits (padrão do Numpy); precisão: ~15 dígitos
- `int64`: inteiro de 64 bits; precisão: ~7 dígitos
- `bool`: booleano (filtros e condições)

> **Regra importante**: NumPy sempre promove para o tipo mais "seguro" (upcasting).

In [ ]:
# NumPy infere o dtype automaticamente a partir dos valores
inteiros = np.array([1, 2, 3])        # int64
floats   = np.array([1.0, 2.0, 3.0])  # float64
mistura  = np.array([1, 2.0, 3])      # float64 (upcasting)

print(f"Array de inteiros: {inteiros.dtype}")
print(f"Array de floats: {floats.dtype}")
print(f"Array misturado: {mistura.dtype}")

# Truncagem silenciosa ao converter para inteiro (não arredonda)
precos = np.array([38.72, 67.50, 32.15])
print(f"\nfloat64: {precos}")
print(f"int32:   {precos.astype(np.int32)} (casas decimais truncadas)")

# Controle explícito com dtype
taxas_f32 = np.array([0.05, 0.10, 0.15], dtype=np.float32)  # 4 bytes/elemento
taxas_f64 = np.array([0.05, 0.10, 0.15], dtype=np.float64)  # 8 bytes (padrão)
print(f"\nfloat32: {taxas_f32}  ({taxas_f32.nbytes} bytes)")
print(f"float64: {taxas_f64}  ({taxas_f64.nbytes} bytes)")

## Criação de arrays <a id="criacao"></a>

Além de `np.array()`, o NumPy oferece diversas funções para criar arrays com padrões específicos — o que é mais rápido e expressivo do que construir listas manualmente.

O argumento shape define o formato do array:

- um inteiro: array 1D
- uma tupla `(linhas, colunas)`: array 2D

In [ ]:
# Zeros, uns e constante
...

# Sequências numéricas: arange() e linspace()
...

# Espaçamento linear (preferir para floats!)
...

**`np.empty`: desempenho com responsabilidade**

A função `np.empty(shape)` cria um array sem inicializar os valores. Ela apenas reserva o espaço na memória.

Isso significa que o array conterá valores aleatórios — resíduos do que já estava naquele espaço (*lixo de memória*).

**Quando usar:**

Use `np.empty` somente se você for preencher todos os elementos imediatamente.

Nesse caso, você evita o custo desnecessário de inicializar com zeros (`np.zeros`), ganhando desempenho.

**Quando evitar:** 

Não use `np.empty` se houver qualquer chance de ler valores antes de preenchê-los — os dados são imprevisíveis.

In [ ]:
# np.empty: reserva memória sem inicializar
...

# Calcular retornos percentuais a partir de preços
#precos = np.array([100., 102., 99., 105., 108., 107.])
#retornos = np.empty(len(precos) - 1)  # aloca sem inicializar

#for i in range(len(retornos)):
#    retornos[i] = (precos[i+1] - precos[i]) / precos[i] * 100

#print("\nRetornos (%):", np.round(retornos, 2))

## Arrays 2D (matrizes) <a id="arrays2d"></a>

Um array 2D é uma estrutura com duas dimensões — linhas e colunas — formando uma grade retangular de números do mesmo tipo.

Na prática, ele é equivalente a uma tabela de dados ou, em linguagem matemática, uma matriz.

**Como criar um array 2D**

Para criar um array 2D, você passa uma lista de listas para `np.array()` — cada lista interna representa uma linha:

```python
dados = np.array([
    [linha_0_col_0,  linha_0_col_1],   # linha 0
    [linha_1_col_0,  linha_1_col_1],   # linha 1
])
```

> 💡 Todas as linhas devem ter o mesmo número de elementos, caso contrário o NumPy não conseguirá formar uma matriz válida.

In [ ]:
# Criação de um array 2D a partir de lista de listas
...

# Matriz identidade (1 na diagonal principal, 0 no restante)
...

# Matriz diagonal (valores definidos na diagonal principal)
...

# Matriz transposta
...

In [ ]:
# Linhas: países | Colunas: PIB per capita (US$ mil), Inflação (%), Desemprego (%)
paises = ["Brasil", "Chile", "Argentina", "México", "Peru"]

dados_paises = np.array([
    [ 9.0,   4.83, 6.2],   # Brasil
    [16.5,   3.20, 8.9],   # Chile
    [11.6, 211.00, 7.7],   # Argentina
    [10.8,   4.70, 2.7],   # México
    [ 6.6,   2.90, 7.4],   # Peru
])

print(dados_paises)
print()

# Propriedades do array 2D
print("shape:  ", dados_paises.shape)   # (5, 3): 5 países, 3 indicadores
print("ndim:   ", dados_paises.ndim)    # 2 dimensões (matriz)
print("size:   ", dados_paises.size)    # 15 valores no total
print("nbytes: ", dados_paises.nbytes, "bytes")

## Indexação e fatiamento (*slicing*) <a id="indexacao"></a>

A indexação no NumPy é uma extensão direta do que você já conhece de listas em Python — só que generalizada para múltiplas dimensões.

**Indexação em 1D**

Em arrays 1D,o comportamento é exatamente o mesmo das listas:

- o índice inicial é inclusivo
- o final é exclusivo
- índices negativos contam a partir do fim

In [ ]:
# Arrays 1D
ipca = np.array([0.42, 0.83, 0.16, 0.38, 0.44, 0.50,
                 0.38, 0.44, 0.56, 0.61, 0.39, 0.52])

print("Primeiro:          ", ...)
print("Último:            ", ...)
print("Índices 3 a 6:     ", ...)
print("Primeiro semestre: ", ...)
print("Segundo semestre:  ", ...)
print("Meses pares:       ", ...)

**Indexação em 2D**

Em arrays 2D (ou mais), cada dimensão é acessada separadamente: `array[linha, coluna]`

> ⚠️ Note o uso de uma vírgula dentro de um único par de colchetes — isso é essencial.

Você pode aplicar fatiamento (*slicing*) em cada eixo de forma independente, o que permite selecionar linhas, colunas ou submatrizes com facilidade.

| Sintaxe     | Retorna                           | Intuição       |
|-------------|-----------------------------------|----------------|
| `a[i, j]`   | elemento na linha `i`, coluna `j` | um único valor |
| `a[i, :]`   | toda a linha `i`                  | um registro completo |
| `a[:, j]`   | toda a coluna `j`                 | uma variável ao longo das observações |
| `a[i:k, :]` | linhas de `i` até `k-1`           | subconjunto de dados |

In [ ]:
# linhas = países, colunas = indicadores
dados = np.array([
    [1.8, 5.2, 10.1],   # País A
    [2.3, 6.1, 9.5],    # País B
    [0.9, 4.8, 11.0]    # País C
])

print(...)   # inflação do País A
print(...)   # todos indicadores do País B
print(...)   # desemprego de todos os países

In [ ]:
# Array 2D: [linha, coluna]
paises = ["Brasil", "Chile", "Argentina", "México", "Peru"]

# Colunas: PIB per capita (US$ mil), Inflação (%), Desemprego (%)
dados_paises = np.array([
    [ 9.0,   4.83, 6.2],   # Brasil
    [16.5,   3.20, 8.9],   # Chile
    [11.6, 211.00, 7.7],   # Argentina
    [10.8,   4.70, 2.7],   # México
    [ 6.6,   2.90, 7.4],   # Peru
])

print(...)    # inflação do Brasil
print(...)    # todos os indicadores do Chile
print(...)    # desemprego de todos os países
print(...)  # Chile e Argentina: inflação e desemprego

**Indexação booleana: filtrando dados sem `for`**

A indexação booleana permite selecionar elementos de um array com base em uma condição lógica — de forma vetorizada, sem precisar de loops.

No exemplo a seguir:

- `dados_paises[:, 1]` extrai a coluna de inflação (todos os países)
- `inflacao > 5.0` cria uma máscara booleana — um array de `True` e `False` indicando quais valores satisfazem a condição
- `dados_paises[mascara]` aplica essa máscara ao array original, retornando apenas as linhas onde a condição é verdadeira e descartando as demais

> 💡 Intuição: a máscara funciona como um filtro que “liga” (True) ou “desliga” (False) cada linha do array.

## Views vs. cópias <a id="views"></a>

Ao fatiar (*slicing*) um array no NumPy, você geralmente não cria um novo array independente — cria uma view (uma "janela") para os mesmos dados na memória.

Isso significa que view e array original compartilham os mesmos valores.

```
array original: [0.42, 0.83, 0.16, ...]
                  ↑     ↑     ↑
                 view aponta para os mesmos dados
```

**Consequência importante**

Se você modificar a view, o array original também será alterado!

In [ ]:
ipca = np.array([0.42, 0.83, 0.16, 0.38, 0.44, 0.50])

# View (mesma memória)
...

print("view alterada:", ...)  # 999.0
print("ipca alterado:", ...)  # mudou!

**Quando usar `copy()`**

Se você precisa de um array independente, use `copy()` para criar uma cópia independente. 

As alterações feitas na cópia não afetam o array original.

In [ ]:
ipca = np.array([0.42, 0.83, 0.16, 0.38, 0.44, 0.50])

# Cópia (memória independente)
...

print("copia alterada:", ...)   # 999.0
print("ipca preservado:", ...)   # 0.42 (não mudou)

## Operações vetorizadas <a id="vetorizadas"></a>

**Vetorização** significa aplicar uma operação a todos os elementos de um array de uma só vez, sem usar `for`.

No NumPy, essas operações são executadas por rotinas otimizadas em C, diretamente sobre blocos contíguos de memória — o que as torna muito mais rápidas e eficientes.

```python
# Python puro (loop explícito)
resultado = [x * 1.05 for x in precos]

# NumPy (vetorizado)
resultado = precos * 1.05
```

**Operações vetorizadas básicas**

Qualquer operação aritmética entre arrays (ou array e escalar) é vetorizada, ou seja, aplicada elemento a elemento:

- `+`, `-`, `*`, `/`, `**`
- funções como `np.sqrt`, `np.log`, `np.exp`

> 💡 Em geral, NumPy pode ser dezenas de vezes mais rápido — especialmente em grandes volumes de dados.

In [ ]:
precos = np.array([100, 102, 99, 105, 110])
taxa_inflacao  = np.array([0.05, 0.10, 0.08, 0.06, 0.09])

print("Reajuste de 10%:    ", ...)
print("Aplicando Inflação: ", ...)
print("Desconto de R$5:    ", ...)

In [ ]:
import time
n = 1_000_000
lista = list(range(n))
arr   = np.arange(n, dtype=float)

t0 = time.perf_counter()
resultado_py = [x**2 for x in lista]
t1 = time.perf_counter()

t2 = time.perf_counter()
resultado_np = arr**2
t3 = time.perf_counter()

print(f"Python puro: {(t1-t0)*1000:.1f} ms")
print(f"NumPy:       {(t3-t2)*1000:.1f} ms")
print(f"NumPy é {(t1-t0)/(t3-t2):.0f}x mais rápido")

In [ ]:
# Inflação acumulada
ipca_mensal = np.array([0.42, 0.83, 0.16, 0.38, 0.44, 0.50,
                        0.38, 0.44, 0.56, 0.61, 0.39, 0.52])

fatores   = 1 + ipca_mensal / 100
acumulado = np.cumprod(fatores) - 1  # acumulado até cada mês
print(acumulado)

meses = ["Jan","Fev","Mar","Abr","Mai","Jun",
         "Jul","Ago","Set","Out","Nov","Dez"]

print(f"\n{'Mês':<6} {'IPCA':>8} {'Acumulado':>12}")
print("─" * 30)
for mes, taxa, acum in zip(meses, ipca_mensal, acumulado * 100):
    print(f"{mes:<6} {taxa:>7.2f}% {acum:>10.2f}%")
print(f"\nInflação anual: {acumulado[-1]*100:.2f}%")

## Funções estatísticas <a id="estatisticas"></a>

O NumPy oferece diversas funções estatísticas que operam diretamente sobre arrays — como `mean`, `sum`, `std`, `min`, `max`, entre outras.

Em arrays 1D, o comportamento é direto. Já em arrays 2D (ou mais), usamos o parâmetro `axis` para definir em qual direção a operação será aplicada.

**Entendendo o axis**

O axis indica qual dimensão a função será aplicada:

- `axis=0`: percorre para baixo (linhas), ou seja, resultado por coluna
- `axis=1`: percorre para a direita (colunas), ou seja, resultado por linha

In [ ]:
# Array 1D
precos = np.array([38.72, 67.50, 32.15, 13.88, 45.20, 52.30, 28.90])

print(f"Média:         ...")
print(f"Mediana:       ...")
print(f"Desvio padrão: ...")
print(f"Variância:     ...")
print(f"Min:           ...")
print(f"Índice do min: ...")
print(f"Max:           ...")
print(f"Índice do max: ...")

In [ ]:
# Array 2D
dados = np.array([
    [10, 20, 30],
    [40, 50, 60],
    [70, 80, 90]
])

print(...)  # média por coluna [40. 50. 60.]
print()
print(...)  # média por linha [20. 50. 80.]

In [ ]:
# axis=0 → por coluna (indicador) | axis=1 → por linha (país)
indicadores = ["PIB per capita", "Inflação", "Desemprego"]

print("Média por indicador (axis=0):")
for ind, val in zip(indicadores, np.mean(dados_paises, axis=0)):
    print(f"  {ind:<18}: {val:.2f}")

print("\nMédia por país (axis=1):")
for p, val in zip(paises, np.mean(dados_paises, axis=1)):
    print(f"  {p:<12}: {val:.2f}")

### Percentis e quantis

Percentis e quantis são ferramentas fundamentais para analisar distribuições — muito usados em economia e finanças para estudar renda, risco e dispersão dos dados.

- Percentil p: valor abaixo do qual estão p% dos dados
- Quantis: generalização dos percentis (ex.: quartis dividem em 4 partes)

In [ ]:
# Distribuição de renda simulada (em R$ mil)
rng = np.random.default_rng(seed = 42)
renda = np.sort(rng.lognormal(mean = 3.0, sigma = 0.8, size = 1000))

# Percentis clássicos
p25, p50, p75 = np.percentile(renda, [25, 50, 75])
print(f"P25 (1º quartil): R$ {p25:,.1f} mil")
print(f"P50 (mediana):    R$ {p50:,.1f} mil")
print(f"P75 (3º quartil): R$ {p75:,.1f} mil")
print(f"IQR:              R$ {p75 - p25:,.1f} mil")

# np.quantile é equivalente com q entre 0 e 1
print(f"\nQuantil 0.5: R$ {np.quantile(renda, 0.5):,.1f} mil")
print(f"\nQuantil 0.9: R$ {np.quantile(renda, 0.9):,.1f} mil")

## Reshape e manipulação de forma <a id="reshape"></a>

No NumPy, você pode alterar a forma (*shape*) de um array sem mudar seus dados — apenas como eles são organizados na memória.

A função reshape permite transformar um array mantendo o mesmo número total de elementos:

In [ ]:
# reshape: o produto das dimensões deve ser igual ao total de elementos
...

# Você pode deixar o NumPy calcular uma dimensão automaticamente:
...

# ravel e flatten: voltando para 1D
...
